In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import warnings, time, os

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)
os.makedirs('/kaggle/working/figures', exist_ok=True)
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Prints every file
for root, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        fp = os.path.join(root, f)
        print(f"{fp}  ({os.path.getsize(fp)/1e9:.2f} GB)")

In [ ]:

# Picked the file called N-CMAPSS_DS02-006.h5


H5_PATH = '/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS02-006.h5' 

# Open and see what's inside
with h5py.File(H5_PATH, 'r') as f:
    print("Keys in file:")
    for key in f.keys():
        shape = f[key].shape if hasattr(f[key], 'shape') else 'group'
        print(f"  {key}: {shape}")

In [ ]:
# N-CMAPSS stores data with _dev (training) and _test (testing) suffixes
# Variable groups: W (operating), X_s (sensors), X_v (virtual), T (health params), Y (RUL), A (auxiliary)

with h5py.File(H5_PATH, 'r') as f:
    # Training data
    W_dev  = np.array(f['W_dev'])             # Operating conditions (4 vars: alt, Mach, TRA, T2)
    X_s_dev = np.array(f['X_s_dev'])          # Physical sensors (13 vars)
    X_v_dev = np.array(f['X_v_dev'])          # Virtual sensors (18 vars) - optional, can skip
    Y_dev  = np.array(f['Y_dev'])             # RUL labels
    A_dev  = np.array(f['A_dev'])             # Unit ID and cycle info
    
    # Test data
    W_test  = np.array(f['W_test'])
    X_s_test = np.array(f['X_s_test'])
    X_v_test = np.array(f['X_v_test'])
    Y_test  = np.array(f['Y_test'])
    A_test  = np.array(f['A_test'])

print(f"W_dev:   {W_dev.shape}  (operating conditions)")
print(f"X_s_dev: {X_s_dev.shape}  (physical sensors)")
print(f"X_v_dev: {X_v_dev.shape}  (virtual sensors)")
print(f"Y_dev:   {Y_dev.shape}  (RUL)")
print(f"A_dev:   {A_dev.shape}  (unit/cycle info)")

# Combining features: operating + physical sensors
# (skipped virtual sensors to keep things simpler and faster)
X_train_raw = np.concatenate([W_dev, X_s_dev], axis=1)
X_test_raw  = np.concatenate([W_test, X_s_test], axis=1)
y_train_raw = Y_dev.ravel()
y_test_raw  = Y_test.ravel()

# Unit IDs
unit_train = A_dev[:, 0].astype(int)
unit_test  = A_test[:, 0].astype(int)

op_names = ['alt', 'Mach', 'TRA', 'T2']
sensor_names = ['Wf', 'Nf', 'Nc', 'T24', 'T30', 'T48', 'T50',
                'P15', 'P21', 'P24', 'Ps30', 'P40', 'P50', 'farB']
feature_names = op_names + sensor_names

n_op = W_dev.shape[1]
n_sensor = X_s_dev.shape[1]
n_features = n_op + n_sensor

print(f"Operating: {n_op} cols, Sensors: {n_sensor} cols, Total: {n_features}")

In [ ]:
# Scale features (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

# Train/val split BY ENGINE
train_engines = np.unique(unit_train)
np.random.seed(42)
np.random.shuffle(train_engines)
n_val = max(1, len(train_engines) // 5)  # 20% for validation
val_engines = set(train_engines[:n_val])

val_mask = np.isin(unit_train, list(val_engines))
X_train = X_train_scaled[~val_mask]
y_train = y_train_raw[~val_mask]
X_val   = X_train_scaled[val_mask]
y_val   = y_train_raw[val_mask]

print(f"Train: {len(X_train):,} samples ({len(train_engines)-n_val} engines)")
print(f"Val:   {len(X_val):,} samples ({n_val} engines)")
print(f"Test:  {len(X_test_scaled):,} samples ({len(np.unique(unit_test))} engines)")
print(f"Val engines: {sorted(val_engines)}")

# Subsampling if dataset is huge (>2M samples makes DL very slow)
MAX_TRAIN = 1_000_000
if len(X_train) > MAX_TRAIN:
    print(f"\nSubsampling train: {len(X_train):,} → {MAX_TRAIN:,}")
    idx = np.random.choice(len(X_train), MAX_TRAIN, replace=False)
    X_train_sub = X_train[idx]
    y_train_sub = y_train[idx]
else:
    X_train_sub = X_train
    y_train_sub = y_train

In [ ]:
# Correlations with RUL
corrs = {}
for i, name in enumerate(feature_names):
    corrs[name] = np.corrcoef(X_train_scaled[:, i], y_train_raw)[0, 1]
corrs = pd.Series(corrs).sort_values(key=abs, ascending=False)
print("Top 10 features correlated with RUL:")
print(corrs.head(10).to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(14, 12))
corr_df = pd.DataFrame(X_train_scaled[:5000], columns=feature_names)  # subsample for speed
corr_df['RUL'] = y_train_raw[:5000]
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, annot_kws={'size':7})
ax.set_title('Correlation Matrix (N-CMAPSS)')
plt.savefig('/kaggle/working/figures/correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Correlation bar chart
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#2166ac' if v > 0 else '#b2182b' for v in corrs.values]
ax.barh(range(len(corrs)), abs(corrs.values), color=colors)
ax.set_yticks(range(len(corrs))); ax.set_yticklabels(corrs.index)
ax.set_xlabel('|Correlation with RUL|'); ax.invert_yaxis()
ax.set_title('Feature Correlation with RUL')
plt.savefig('/kaggle/working/figures/feature_corr_bar.png', bbox_inches='tight')
plt.show()

# Sensor vs operating category
sensor_corrs = [abs(corrs[s]) for s in sensor_names if s in corrs]
op_corrs = [abs(corrs[s]) for s in op_names if s in corrs]
print(f"\nMean |r| by category:")
print(f"  Sensors ({len(sensor_names)}):   {np.mean(sensor_corrs):.4f}")
print(f"  Operating ({len(op_names)}): {np.mean(op_corrs):.4f}")

In [ ]:
t0 = time.time()
lr_model = LinearRegression().fit(X_train_sub, y_train_sub)
lr_time = time.time() - t0

y_pred_lr = lr_model.predict(X_test_scaled)
lr_rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_lr))
lr_mae  = mean_absolute_error(y_test_raw, y_pred_lr)
lr_r2   = r2_score(y_test_raw, y_pred_lr)
print(f"LR: RMSE={lr_rmse:.2f}, MAE={lr_mae:.2f}, R²={lr_r2:.4f}, Time={lr_time:.2f}s")

fig, ax = plt.subplots(figsize=(7, 7))
s = np.random.choice(len(y_test_raw), min(5000, len(y_test_raw)), replace=False)
ax.scatter(y_test_raw[s], y_pred_lr[s], alpha=0.1, s=3)
ax.plot([y_test_raw.min(), y_test_raw.max()], [y_test_raw.min(), y_test_raw.max()], 'r--')
ax.set_xlabel('Actual RUL'); ax.set_ylabel('Predicted RUL')
ax.set_title(f'Linear Regression: RMSE={lr_rmse:.2f}, R²={lr_r2:.4f}')
plt.savefig('/kaggle/working/figures/lr_scatter.png')
plt.show()

In [ ]:
print("Grid search:")
results = []
for md in [4, 6, 8]:
    for lr in [0.05, 0.1, 0.2]:
        m = xgb.XGBRegressor(n_estimators=300, learning_rate=lr, max_depth=md,
            subsample=0.8, colsample_bytree=0.8, tree_method='hist', device='cuda',
            random_state=42, early_stopping_rounds=20, eval_metric='rmse')
        m.fit(X_train_sub, y_train_sub, eval_set=[(X_val, y_val)], verbose=0)
        pred = m.predict(X_test_scaled)
        rmse = np.sqrt(mean_squared_error(y_test_raw, pred))
        results.append({'depth': md, 'lr': lr, 'rmse': rmse, 'iters': m.best_iteration})
        print(f"  depth={md}, lr={lr}: RMSE={rmse:.2f}")

rdf = pd.DataFrame(results)
best = rdf.loc[rdf['rmse'].idxmin()]
print(f"Best: depth={int(best['depth'])}, lr={best['lr']}, RMSE={best['rmse']:.2f}")
rdf.to_csv('/kaggle/working/figures/hyperparam_search.csv', index=False)

# Training best model
t0 = time.time()
xgb_model = xgb.XGBRegressor(n_estimators=300, learning_rate=best['lr'],
    max_depth=int(best['depth']), subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=20, eval_metric='rmse')
xgb_model.fit(X_train_sub, y_train_sub,
              eval_set=[(X_train_sub, y_train_sub), (X_val, y_val)], verbose=50)
xgb_time = time.time() - t0

y_pred_xgb = xgb_model.predict(X_test_scaled)
xgb_rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_xgb))
xgb_mae  = mean_absolute_error(y_test_raw, y_pred_xgb)
xgb_r2   = r2_score(y_test_raw, y_pred_xgb)
print(f"\nXGBoost: RMSE={xgb_rmse:.2f}, MAE={xgb_mae:.2f}, R²={xgb_r2:.4f}, Time={xgb_time:.1f}s")
print(f"vs LR: {(1-xgb_rmse/lr_rmse)*100:+.1f}%")

# 5-fold CV (engine-level)
print("\n5-Fold CV:")
gkf = GroupKFold(n_splits=5)
cv = []
for fold, (tr, va) in enumerate(gkf.split(X_train_scaled, y_train_raw, unit_train)):
    m = xgb.XGBRegressor(n_estimators=300, learning_rate=best['lr'],
        max_depth=int(best['depth']), subsample=0.8, colsample_bytree=0.8,
        tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=20, eval_metric='rmse')
    m.fit(X_train_scaled[tr], y_train_raw[tr], eval_set=[(X_train_scaled[va], y_train_raw[va])], verbose=0)
    fold_rmse = np.sqrt(mean_squared_error(y_train_raw[va], m.predict(X_train_scaled[va])))
    cv.append(fold_rmse)
    print(f"  Fold {fold+1}: {fold_rmse:.2f}")
print(f"  Mean: {np.mean(cv):.2f} ± {np.std(cv):.2f}")

# Figures
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
s = np.random.choice(len(y_test_raw), min(5000, len(y_test_raw)), replace=False)
axes[0].scatter(y_test_raw[s], y_pred_xgb[s], alpha=0.1, s=3)
axes[0].plot([y_test_raw.min(),y_test_raw.max()],[y_test_raw.min(),y_test_raw.max()],'r--')
axes[0].set_title(f'XGBoost: RMSE={xgb_rmse:.2f}, R²={xgb_r2:.4f}')

ev = xgb_model.evals_result()
axes[1].plot(ev['validation_0']['rmse'], label='Train')
axes[1].plot(ev['validation_1']['rmse'], label='Val')
axes[1].legend(); axes[1].set_title('Learning Curves')

imp = xgb_model.feature_importances_
top = np.argsort(imp)[-15:]
axes[2].barh(range(len(top)), imp[top])
axes[2].set_yticks(range(len(top))); axes[2].set_yticklabels([feature_names[i] for i in top])
axes[2].set_title('Feature Importance')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/xgboost_results.png')
plt.show()

In [ ]:
WINDOW = 50  # 50 timesteps (standard for N-CMAPSS at 1Hz)

# N-CMAPSS large. Subsample to 0.1Hz (every 10th sample) to fit in memory
SUBSAMPLE = 10

def make_windows(X, y, units, window=WINDOW, subsample=SUBSAMPLE):
    Xw, yw = [], []
    for uid in np.unique(units):
        mask = units == uid
        feat = X[mask][::subsample]  # subsample
        rul = y[mask][::subsample]
        for i in range(len(feat) - window + 1):
            Xw.append(feat[i:i+window])
            yw.append(rul[i + window - 1])
    return np.array(Xw, dtype=np.float32), np.array(yw, dtype=np.float32)

print(f"Window={WINDOW}, Subsample=1/{SUBSAMPLE}")
print("Creating windows (this may take a minute)...")

X_tr_w, y_tr_w = make_windows(X_train_scaled[~val_mask], y_train_raw[~val_mask], unit_train[~val_mask])
X_va_w, y_va_w = make_windows(X_train_scaled[val_mask], y_train_raw[val_mask], unit_train[val_mask])
X_te_w, y_te_w = make_windows(X_test_scaled, y_test_raw, unit_test)

print(f"Train: {X_tr_w.shape} | Val: {X_va_w.shape} | Test: {X_te_w.shape}")
print(f"Memory: ~{(X_tr_w.nbytes + X_va_w.nbytes + X_te_w.nbytes) / 1e9:.2f} GB")

# If still too big, subsample training windows
MAX_WIN = 300_000
if len(X_tr_w) > MAX_WIN:
    print(f"Subsampling windows: {len(X_tr_w):,} → {MAX_WIN:,}")
    idx = np.random.choice(len(X_tr_w), MAX_WIN, replace=False)
    X_tr_w, y_tr_w = X_tr_w[idx], y_tr_w[idx]

In [ ]:
cnn = keras.Sequential([
    layers.Input(shape=(WINDOW, n_features)),
    layers.Conv1D(64, 5, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.MaxPooling1D(2), layers.Dropout(0.2),
    layers.Conv1D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.MaxPooling1D(2), layers.Dropout(0.2),
    layers.Conv1D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.GlobalAveragePooling1D(), layers.Dropout(0.3),
    layers.Dense(64, activation='relu'), layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='linear')
])
cnn.compile(optimizer=optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
cnn.summary()

t0 = time.time()
h_cnn = cnn.fit(X_tr_w, y_tr_w, validation_data=(X_va_w, y_va_w),
                epochs=50, batch_size=256,
                callbacks=[callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
                           callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=5)],
                verbose=1)
cnn_time = time.time() - t0

y_pred_cnn = cnn.predict(X_te_w, batch_size=256).ravel()
cnn_rmse = np.sqrt(mean_squared_error(y_te_w, y_pred_cnn))
cnn_mae  = mean_absolute_error(y_te_w, y_pred_cnn)
cnn_r2   = r2_score(y_te_w, y_pred_cnn)

print(f"\n1D-CNN: RMSE={cnn_rmse:.2f}, MAE={cnn_mae:.2f}, R²={cnn_r2:.4f}")
print(f"Params={cnn.count_params():,}, Time={cnn_time:.0f}s, Epochs={len(h_cnn.history['loss'])}")
print(f"vs XGBoost: {(1-cnn_rmse/xgb_rmse)*100:+.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(h_cnn.history['loss'], label='Train'); axes[0].plot(h_cnn.history['val_loss'], label='Val')
axes[0].legend(); axes[0].set_yscale('log'); axes[0].set_title('1D-CNN Loss')
s = np.random.choice(len(y_te_w), min(5000, len(y_te_w)), replace=False)
axes[1].scatter(y_te_w[s], y_pred_cnn[s], alpha=0.1, s=3, c='darkorange')
axes[1].plot([y_te_w.min(),y_te_w.max()],[y_te_w.min(),y_te_w.max()],'r--')
axes[1].set_title(f'1D-CNN: RMSE={cnn_rmse:.2f}, R²={cnn_r2:.4f}')
r = y_te_w - y_pred_cnn
axes[2].hist(r, bins=80, color='darkorange'); axes[2].axvline(0, color='red', ls='--')
axes[2].set_title(f'Residuals: mean={r.mean():.2f}, std={r.std():.2f}')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/cnn_results.png')
plt.show()

In [ ]:
cl = keras.Sequential([
    layers.Input(shape=(WINDOW, n_features)),
    layers.Conv1D(64, 5, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.MaxPooling1D(2), layers.Dropout(0.2),
    layers.Conv1D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.MaxPooling1D(2), layers.Dropout(0.2),
    layers.LSTM(64, return_sequences=True), layers.Dropout(0.2),
    layers.LSTM(32, return_sequences=False), layers.Dropout(0.2),
    layers.Dense(32, activation='relu'), layers.Dropout(0.2),
    layers.Dense(1, activation='linear')
])
cl.compile(optimizer=optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
cl.summary()

t0 = time.time()
h_cl = cl.fit(X_tr_w, y_tr_w, validation_data=(X_va_w, y_va_w),
              epochs=50, batch_size=256,
              callbacks=[callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
                         callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=5)],
              verbose=1)
cl_time = time.time() - t0

y_pred_cl = cl.predict(X_te_w, batch_size=256).ravel()
cl_rmse = np.sqrt(mean_squared_error(y_te_w, y_pred_cl))
cl_mae  = mean_absolute_error(y_te_w, y_pred_cl)
cl_r2   = r2_score(y_te_w, y_pred_cl)

print(f"\nCNN-LSTM: RMSE={cl_rmse:.2f}, MAE={cl_mae:.2f}, R²={cl_r2:.4f}")
print(f"Params={cl.count_params():,}, Time={cl_time:.0f}s, Epochs={len(h_cl.history['loss'])}")
print(f"vs XGBoost: {(1-cl_rmse/xgb_rmse)*100:+.1f}%")
print(f"vs CNN:     {(1-cl_rmse/cnn_rmse)*100:+.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(h_cl.history['loss'], label='Train'); axes[0].plot(h_cl.history['val_loss'], label='Val')
axes[0].legend(); axes[0].set_yscale('log'); axes[0].set_title('CNN-LSTM Loss')
s = np.random.choice(len(y_te_w), min(5000, len(y_te_w)), replace=False)
axes[1].scatter(y_te_w[s], y_pred_cl[s], alpha=0.1, s=3, c='forestgreen')
axes[1].plot([y_te_w.min(),y_te_w.max()],[y_te_w.min(),y_te_w.max()],'r--')
axes[1].set_title(f'CNN-LSTM: RMSE={cl_rmse:.2f}, R²={cl_r2:.4f}')
r2 = y_te_w - y_pred_cl
axes[2].hist(r2, bins=80, color='forestgreen'); axes[2].axvline(0, color='red', ls='--')
axes[2].set_title(f'Residuals: mean={r2.mean():.2f}, std={r2.std():.2f}')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/cnn_lstm_results.png')
plt.show()

In [ ]:
print("\n=== MODEL COMPARISON ===")
print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'Time':>10}")
print("-" * 58)
for name, rmse, mae, r2, t in [
    ('Linear Regression', lr_rmse, lr_mae, lr_r2, lr_time),
    ('XGBoost', xgb_rmse, xgb_mae, xgb_r2, xgb_time),
    ('1D-CNN', cnn_rmse, cnn_mae, cnn_r2, cnn_time),
    ('CNN-LSTM', cl_rmse, cl_mae, cl_r2, cl_time)]:
    print(f"{name:<20} {rmse:>8.2f} {mae:>8.2f} {r2:>8.4f} {t:>9.1f}s")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
names = ['LR', 'XGB', 'CNN', 'CNN-LSTM']
c = ['grey', 'steelblue', 'darkorange', 'forestgreen']
rmses = [lr_rmse, xgb_rmse, cnn_rmse, cl_rmse]

axes[0].bar(names, rmses, color=c); axes[0].set_ylabel('RMSE'); axes[0].set_title('RMSE')
for i,v in enumerate(rmses): axes[0].text(i, v+0.3, f'{v:.2f}', ha='center', fontsize=9)
axes[1].bar(names, [lr_r2, xgb_r2, cnn_r2, cl_r2], color=c); axes[1].set_ylabel('R²'); axes[1].set_title('R²')
axes[2].bar(names, [lr_time, xgb_time, cnn_time, cl_time], color=c)
axes[2].set_ylabel('Seconds'); axes[2].set_title('Train Time'); axes[2].set_yscale('log')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/comparison.png')
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_model)
idx = np.random.choice(len(X_test_scaled), min(3000, len(X_test_scaled)), replace=False)
sv = explainer.shap_values(X_test_scaled[idx])

plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_test_scaled[idx], feature_names=feature_names, show=False, max_display=17)
plt.savefig('/kaggle/working/figures/shap_summary.png', bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_test_scaled[idx], feature_names=feature_names, plot_type='bar', show=False)
plt.savefig('/kaggle/working/figures/shap_bar.png', bbox_inches='tight')
plt.show()

ms = np.abs(sv).mean(0); total = ms.sum()
print("SHAP importance:")
for i in np.argsort(ms)[::-1][:10]:
    print(f"  {feature_names[i]:>8s}: {ms[i]:.4f} ({ms[i]/total*100:.1f}%)")

sensor_pct = sum(ms[i] for i,c in enumerate(feature_names) if c in sensor_names) / total * 100
op_pct = 100 - sensor_pct
print(f"\nSensors: {sensor_pct:.1f}% | Operating: {op_pct:.1f}%")

# Temperature sensors specifically
temp_names = ['T2', 'T24', 'T30', 'T48', 'T50']
temp_pct = sum(ms[i] for i, c in enumerate(feature_names) if c in temp_names) / total * 100
print(f"Temperature sensors only: {temp_pct:.1f}%")

In [ ]:
print("\nError by RUL range (XGBoost):")
res = y_test_raw - y_pred_xgb
rul_max = y_test_raw.max()
for label, lo, hi in [('End-of-life (0-10)', 0, 10), ('Mid-life (10-40)', 10, 40), ('Early (40+)', 40, rul_max+1)]:
    mask = (y_test_raw >= lo) & (y_test_raw < hi)
    if mask.sum() > 0:
        rmse = np.sqrt(mean_squared_error(y_test_raw[mask], y_pred_xgb[mask]))
        print(f"  {label}: RMSE={rmse:.2f} (n={mask.sum():,})")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
s = np.random.choice(len(res), min(5000, len(res)), replace=False)
axes[0].scatter(y_test_raw[s], res[s], alpha=0.1, s=3)
axes[0].axhline(0, color='red', ls='--'); axes[0].set_xlabel('True RUL'); axes[0].set_ylabel('Residual')
axes[1].hist(res[s], bins=80); axes[1].axvline(0, color='red', ls='--')
axes[1].set_title(f'mean={res.mean():.2f}, std={res.std():.2f}')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/error_analysis.png')
plt.show()

In [ ]:
cnn.save('/kaggle/working/cnn_model.keras')
cl.save('/kaggle/working/cnn_lstm_model.keras')
xgb_model.save_model('/kaggle/working/xgb_model.json')

print("\n NUMBERS FOR REPORT\n")
print(f"Dataset: N-CMAPSS DS02, {len(y_train_raw)+len(y_test_raw):,} total samples, 1Hz")
print(f"Train: {len(y_train_raw):,} samples, engines {sorted(np.unique(unit_train))}")
print(f"Test:  {len(y_test_raw):,} samples, engines {sorted(np.unique(unit_test))}")
print(f"Features: {n_features} ({len(op_names)} operating + {len(sensor_names)} sensors)")
print(f"\nLinear Regression: RMSE={lr_rmse:.2f}, MAE={lr_mae:.2f}, R²={lr_r2:.4f}")
print(f"XGBoost:           RMSE={xgb_rmse:.2f}, MAE={xgb_mae:.2f}, R²={xgb_r2:.4f}")
print(f"  Best config: depth={int(best['depth'])}, lr={best['lr']}")
print(f"  5-Fold CV:   {np.mean(cv):.2f} ± {np.std(cv):.2f}")
print(f"1D-CNN:            RMSE={cnn_rmse:.2f}, MAE={cnn_mae:.2f}, R²={cnn_r2:.4f}, params={cnn.count_params():,}")
print(f"CNN-LSTM:          RMSE={cl_rmse:.2f}, MAE={cl_mae:.2f}, R²={cl_r2:.4f}, params={cl.count_params():,}")
print(f"\nXGBoost vs LR:      {(1-xgb_rmse/lr_rmse)*100:+.1f}%")
print(f"1D-CNN vs XGBoost:  {(1-cnn_rmse/xgb_rmse)*100:+.1f}%")
print(f"CNN-LSTM vs XGBoost:{(1-cl_rmse/xgb_rmse)*100:+.1f}%")

print(f"\nFigures saved:")
for f in sorted(os.listdir('/kaggle/working/figures/')): print(f"  {f}")
print("\nDone.")

In [ ]:
# Fair comparison: XGBoost on same test data as DL models
xgb_win_preds = xgb_model.predict(X_te_w[:, -1, :])
xgb_win_rmse = np.sqrt(mean_squared_error(y_te_w, xgb_win_preds))
print(f"XGBoost on windowed test: RMSE={xgb_win_rmse:.2f}")
print(f"CNN still better: {(1-cnn_rmse/xgb_win_rmse)*100:+.1f}%")